# DART 공시 보고서 수집
삼양그룹 계열사의 사업보고서 / 반기보고서 / 분기보고서 / 감사보고서 수집 (최대 6년)

- 수집 대상: 사업보고서(A001), 반기보고서(A002), 분기보고서(A003), 감사보고서(F001), 연결감사보고서(F002)
- API 문서: https://opendart.fss.or.kr/guide/main.do

In [ ]:
import os
import io
import zipfile
import xml.etree.ElementTree as ET
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('DART_API_KEY')
assert API_KEY, 'DART_API_KEY가 .env에 없습니다.'
print('API Key 로드 완료')

## 1. DART corp_code 매핑 로드
DART는 종목코드 대신 자체 `corp_code`를 사용합니다.  
`corpCode.xml` zip을 받아서 회사명으로 corp_code를 찾습니다.

In [ ]:
def fetch_corp_code_df(api_key: str) -> pd.DataFrame:
    """DART 전체 기업 고유번호 목록을 DataFrame으로 반환"""
    url = f'https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={api_key}'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        with z.open('CORPCODE.xml') as f:
            tree = ET.parse(f)

    rows = []
    for item in tree.getroot().findall('list'):
        rows.append({
            'corp_code': item.findtext('corp_code'),
            'corp_name': item.findtext('corp_name'),
            'stock_code': item.findtext('stock_code'),
            'modify_date': item.findtext('modify_date'),
        })
    return pd.DataFrame(rows)

corp_df = fetch_corp_code_df(API_KEY)
print(f'전체 기업 수: {len(corp_df):,}')
corp_df.head(3)

## 2. 수집 대상 회사 정의

In [ ]:
# 수집 대상 회사 (회사명: 종목코드 or None)
TARGET_COMPANIES = {
    '삼양사':           '145990',
    '삼양패키징':       '272550',
    '삼양이노켐':       None,
    '삼양엔씨켐':       '482630',
    '삼양웰푸드':       None,
    '삼양바이오팜':     None,
    '삼양애니팜':       None,
    '삼양데이타시스템': None,
}

# 회사명으로 corp_code 매핑
def get_corp_code(corp_name: str, stock_code: str | None, df: pd.DataFrame) -> str | None:
    # 종목코드가 있으면 종목코드로 먼저 조회 (더 정확)
    if stock_code:
        matched = df[df['stock_code'] == stock_code]
        if not matched.empty:
            return matched.iloc[0]['corp_code']
    # 회사명 정확 일치
    matched = df[df['corp_name'] == corp_name]
    if not matched.empty:
        return matched.iloc[0]['corp_code']
    # 회사명 부분 일치 (fallback)
    matched = df[df['corp_name'].str.contains(corp_name, na=False)]
    if not matched.empty:
        print(f'  [{corp_name}] 부분 일치 후보: {matched["corp_name"].tolist()}')
        return matched.iloc[0]['corp_code']
    return None

company_map = {}  # corp_name -> corp_code
for name, stock in TARGET_COMPANIES.items():
    code = get_corp_code(name, stock, corp_df)
    company_map[name] = code
    status = code if code else '못 찾음'
    print(f'{name:20s} corp_code: {status}')

## 3. 공시 목록 수집 함수

In [ ]:
REPORT_TYPES = {
    'A001': '사업보고서',
    'A002': '반기보고서',
    'A003': '분기보고서',
    'F001': '감사보고서',
    'F002': '연결감사보고서',
}

# 6년치 날짜 범위
END_DATE   = datetime.today().strftime('%Y%m%d')
START_DATE = str(int(END_DATE[:4]) - 6) + END_DATE[4:]
print(f'수집 기간: {START_DATE} ~ {END_DATE}')


def fetch_report_list(api_key: str, corp_code: str, report_type: str,
                      bgn_de: str, end_de: str) -> list[dict]:
    """특정 회사의 특정 보고서 공시 목록 반환 (페이지 자동 순회)"""
    base_url = 'https://opendart.fss.or.kr/api/list.json'
    results = []
    page = 1

    while True:
        params = {
            'crtfc_key':        api_key,
            'corp_code':        corp_code,
            'pblntf_detail_ty': report_type,
            'bgn_de':           bgn_de,
            'end_de':           end_de,
            'page_no':          page,
            'page_count':       100,
        }
        resp = requests.get(base_url, params=params, timeout=20)
        resp.raise_for_status()
        data = resp.json()

        if data.get('status') != '000':
            # 000=정상, 013=조회된 데이터가 없음
            break

        results.extend(data.get('list', []))

        total_count = int(data.get('total_count', 0))
        if len(results) >= total_count:
            break
        page += 1

    return results

## 4. 전체 수집 실행

In [ ]:
all_records = []

for corp_name, corp_code in company_map.items():
    if not corp_code:
        print(f'[SKIP] {corp_name}: corp_code 없음')
        continue

    for rtype, rname in REPORT_TYPES.items():
        items = fetch_report_list(API_KEY, corp_code, rtype, START_DATE, END_DATE)
        for item in items:
            item['corp_name_kr'] = corp_name
            item['report_type_name'] = rname
        all_records.extend(items)
        print(f'  {corp_name} | {rname}: {len(items)}건')

print(f'\n총 수집: {len(all_records)}건')

In [ ]:
# API 응답 구조 확인 - 삼양사 사업보고서 1건만 테스트
test_corp_name, test_corp_code = '삼양사', company_map['삼양사']
params = {
    'crtfc_key':        API_KEY,
    'corp_code':        test_corp_code,
    'pblntf_detail_ty': 'A001',
    'bgn_de':           START_DATE,
    'end_de':           END_DATE,
    'page_no':          1,
    'page_count':       1,
}
raw = requests.get('https://opendart.fss.or.kr/api/list.json', params=params, timeout=20).json()

print('=== 응답 최상위 키 ===')
for key, val in raw.items():
    if key != 'list':
        print(f'  {key:15s}: {val}')

print()
print('=== list[0] 필드 (레코드 1건) ===')
if raw.get('list'):
    for key, val in raw['list'][0].items():
        print(f'  {key:20s}: {val}')

## 4-1. 보고서 본문 XML 수집

`rcept_no`로 `/document.xml` 호출 → ZIP 안의 XML 파일을 파싱합니다.  
ZIP 안에는 보고서 섹션별 XML 파일이 여러 개 들어있으며, 확장자 없는 파일이 본문 원본입니다.